# ROGII Wellbore Geology Prediction - Kaggle Submission

This notebook generates the final submission file for the Kaggle competition.

**Important**: This notebook must run with internet disabled for Kaggle submission.

## Workflow
1. Load test data
2. Apply same preprocessing as training
3. Load trained model
4. Generate predictions
5. Create submission.csv
6. Validate submission format

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from src.data import load_all_data
from src.preprocess import preprocess_data, add_features
from src.model import load_model
from src.utils import validate_submission, set_random_seed
from config import DATA_DIR, MODELS_DIR, RANDOM_SEED

# Set random seed for reproducibility
set_random_seed(RANDOM_SEED)

print("✓ Setup complete")

## Load Test Data

In [ ]:
print("Loading test data...")
test_df = load_all_data(str(DATA_DIR), 'test')

print(f"Test data shape: {test_df.shape}")
print(f"Number of test wells: {test_df['well_id'].nunique()}")
print(f"Test well IDs: {test_df['well_id'].unique().tolist()}")
print(f"\nTest data preview:")
print(test_df.head())

## Preprocess Test Data

In [ ]:
# Load saved scaler from training
scaler_path = MODELS_DIR / "scaler.pkl"
scaler = joblib.load(scaler_path)
print(f"✓ Loaded scaler from {scaler_path}")

# Apply same preprocessing as training
print("\nPreprocessing test data...")
# Note: Need to modify preprocess_data to accept scaler parameter
# For now, we'll do it manually

from sklearn.impute import SimpleImputer

test_processed = test_df.copy()

# Handle missing values
numeric_cols = test_processed.select_dtypes(include=[np.number]).columns
imputer = SimpleImputer(strategy='median')
test_processed[numeric_cols] = imputer.fit_transform(test_processed[numeric_cols])

# Apply scaler (exclude target and IDs)
exclude_cols = ['well_id', 'id']
numeric_features = [col for col in numeric_cols if col not in exclude_cols]
test_processed[numeric_features] = scaler.transform(test_processed[numeric_features])

print(f"✓ Preprocessing complete")
print(f"  Shape: {test_processed.shape}")

## Feature Engineering

In [ ]:
print("Generating features...")
test_featured = add_features(test_processed)

print(f"✓ Feature engineering complete")
print(f"  Shape: {test_featured.shape}")
print(f"  Features: {test_featured.shape[1]} columns")

## Load Model and Generate Predictions

In [ ]:
# Load best model (change model type as needed)
model_type = 'lgb'  # Change to 'xgb' or 'cat' if using different model
model_path = MODELS_DIR / f"{model_type}_model.pkl"

print(f"Loading model from {model_path}...")
model = load_model(str(model_path))
print(f"✓ Model loaded successfully")

# Prepare features (same as training)
exclude_cols = ['well_id', 'id']
feature_cols = [col for col in test_featured.columns if col not in exclude_cols]

# Handle missing values
test_featured[feature_cols] = test_featured[feature_cols].fillna(0)

X_test = test_featured[feature_cols]

print(f"\nGenerating predictions...")
predictions = model.predict(X_test)

print(f"✓ Predictions generated")
print(f"  Prediction range: [{predictions.min():.4f}, {predictions.max():.4f}]")
print(f"  Prediction mean: {predictions.mean():.4f} ± {predictions.std():.4f}")

## Create Submission File

In [ ]:
# Create submission DataFrame
submission = pd.DataFrame({
    'id': test_featured['id'],
    'tvt': predictions
})

print("Submission preview:")
print(submission.head(10))
print(f"\nSubmission shape: {submission.shape}")

In [ ]:
# Save submission file
submission_path = Path('..') / 'submission.csv'
submission.to_csv(submission_path, index=False)

print(f"✓ Submission saved to {submission_path}")

## Validate Submission

In [ ]:
# Validate submission format
sample_path = Path('..') / 'sample_submission.csv'

try:
    validate_submission(str(submission_path), str(sample_path))
    print("\n✅ Submission is valid and ready for upload!")
except Exception as e:
    print(f"\n❌ Validation error: {e}")

## Submission Statistics

In [ ]:
import matplotlib.pyplot as plt

# Plot prediction distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(predictions, bins=50, alpha=0.7, edgecolor='black')
plt.xlabel('Predicted TVT')
plt.ylabel('Frequency')
plt.title('Distribution of Predictions')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(predictions)
plt.ylabel('Predicted TVT')
plt.title('Prediction Box Plot')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistics by well
submission_with_well = submission.copy()
submission_with_well['well_id'] = test_featured['well_id']

print("\nPredictions by well:")
well_stats = submission_with_well.groupby('well_id')['tvt'].agg(['count', 'mean', 'std', 'min', 'max'])
print(well_stats)

## Optional: Ensemble Predictions

If you have multiple models, you can ensemble them here.

In [ ]:
# Example: Load multiple models and average predictions
# Uncomment and modify as needed

# lgb_model = load_model(str(MODELS_DIR / 'lgb_model.pkl'))
# xgb_model = load_model(str(MODELS_DIR / 'xgb_model.pkl'))
# cat_model = load_model(str(MODELS_DIR / 'cat_model.pkl'))

# lgb_pred = lgb_model.predict(X_test)
# xgb_pred = xgb_model.predict(X_test)
# cat_pred = cat_model.predict(X_test)

# # Simple average
# ensemble_pred = (lgb_pred + xgb_pred + cat_pred) / 3

# # Or weighted average based on CV scores
# # ensemble_pred = 0.4 * lgb_pred + 0.35 * xgb_pred + 0.25 * cat_pred

# submission['tvt'] = ensemble_pred
# submission.to_csv(submission_path, index=False)
# print("✓ Ensemble submission saved")

## Summary

✅ Test data loaded and preprocessed  
✅ Features engineered  
✅ Model loaded  
✅ Predictions generated  
✅ Submission file created and validated  

**Next step**: Upload `submission.csv` to Kaggle!

---

**Kaggle Submission Checklist**:
- [ ] Internet access disabled
- [ ] Runtime < 9 hours
- [ ] Submission file named `submission.csv`
- [ ] Format validated (id, tvt columns)
- [ ] No missing values
- [ ] All test IDs present